# Task 1.1 — Ingest Flight Schedule CSV (MERGE-based Bronze)

**Design decision:** Flight status changes 8–10 times per flight (SCH→BRD→DEP→ARR) — use MERGE not append.

- MERGE INTO `bronze.flight_schedule` ON `flight_id`: update status, delay_minutes, gate on change
- Keep version history: add `effective_ts` column with each MERGE timestamp
- DQ: `flight_id` must be unique per daily schedule; `scheduled_departure` must be valid timestamp
- Referential integrity: `aircraft_registration` must exist in `aircraft_master` table


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Adjust these paths for your environment
DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"

# SparkSession — auto-available on Databricks; create for local
try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_1_1_Flight_Schedule_Bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Raw Ingestion

In [ ]:
# ── Raw Ingestion ─────────────────────────────────────────────────
# Read flight_schedule.csv exactly as-is (absolute raw ingestion)
raw_flight_schedule_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{DATA_DIR}/flight_schedule/flight_schedule.csv"))

# Add ingestion metadata
raw_flight_schedule_df = raw_flight_schedule_df.withColumn(
    "ingestion_ts", F.current_timestamp()
)

print(f"Raw rows ingested: {raw_flight_schedule_df.count()}")
raw_flight_schedule_df.printSchema()
raw_flight_schedule_df.show(5, truncate=False)


## Step 2 — Data Quality & Sanity Checks

In [ ]:
# ── Data Quality & Sanity Checks ──────────────────────────────────

# 1. flight_id uniqueness
dup_ids = raw_flight_schedule_df.groupBy("flight_id").count().filter("count > 1")
print(f"Duplicate flight_ids: {dup_ids.count()}")
if dup_ids.count() > 0:
    dup_ids.show()

# 2. scheduled_departure must be a valid timestamp
invalid_dep = raw_flight_schedule_df.filter(
    F.col("scheduled_departure").isNull() | (F.col("scheduled_departure") == "")
)
print(f"Invalid scheduled_departure records: {invalid_dep.count()}")

# 3. status must be in known enum
VALID_STATUSES = ["SCH", "BRD", "DEP", "ARR", "DLY", "CNX"]
invalid_status = raw_flight_schedule_df.filter(~F.col("status").isin(VALID_STATUSES))
print(f"Invalid status values: {invalid_status.count()}")
if invalid_status.count() > 0:
    invalid_status.select("flight_id", "status").show()

# 4. Build validated dataframe with proper types
validated_flight_schedule_df = (raw_flight_schedule_df
    .withColumn("scheduled_departure", F.to_timestamp("scheduled_departure"))
    .withColumn("scheduled_arrival", F.to_timestamp("scheduled_arrival"))
    .withColumn("delay_minutes", F.col("delay_minutes").cast("int"))
    .withColumn("pax_count_booked", F.col("pax_count_booked").cast("int"))
    .withColumn("effective_ts", F.current_timestamp())
    .filter(F.col("status").isin(VALID_STATUSES))
)

print(f"\nValidated rows: {validated_flight_schedule_df.count()}")
validated_flight_schedule_df.show(5, truncate=False)


## Step 3 — Write to Bronze Delta Table (MERGE on flight_id)

In [ ]:
# ── Write to Bronze Delta Table ───────────────────────────────────
bronze_flight_path = f"{BRONZE_DIR}/flight_schedule"

if DeltaTable.isDeltaTable(spark, bronze_flight_path):
    # MERGE: update status, delay_minutes, gate on change
    bronze_table = DeltaTable.forPath(spark, bronze_flight_path)
    bronze_table.alias("tgt").merge(
        validated_flight_schedule_df.alias("src"),
        "tgt.flight_id = src.flight_id"
    ).whenMatchedUpdate(set={
        "status": "src.status",
        "delay_minutes": "src.delay_minutes",
        "delay_reason_code": "src.delay_reason_code",
        "gate": "src.gate",
        "effective_ts": "src.effective_ts",
        "ingestion_ts": "src.ingestion_ts"
    }).whenNotMatchedInsertAll().execute()
    print(f"MERGE completed on {bronze_flight_path}")
else:
    # Initial load
    (validated_flight_schedule_df.write
        .format("delta")
        .mode("overwrite")
        .save(bronze_flight_path))
    print(f"Initial load completed to {bronze_flight_path}")

# Verify
bronze_flight_df = spark.read.format("delta").load(bronze_flight_path)
print(f"Bronze flight_schedule total rows: {bronze_flight_df.count()}")
bronze_flight_df.show(5, truncate=False)


## Step 4 — Referential Integrity Check

In [ ]:
# ── Referential Integrity: aircraft_registration → aircraft_master
aircraft_master_df = (spark.read
    .option("header", "true")
    .csv(f"{DATA_DIR}/aircraft_master/aircraft_master.csv"))

flight_regs = bronze_flight_df.select("aircraft_registration").distinct()
master_regs = aircraft_master_df.select("aircraft_registration").distinct()

orphan_regs = flight_regs.subtract(master_regs)
print(f"Aircraft registrations WITHOUT master record: {orphan_regs.count()}")
if orphan_regs.count() > 0:
    orphan_regs.show(truncate=False)
else:
    print("All aircraft registrations have valid master records. RI check PASSED.")
